# DARU benchmark — System 1 → System 2 cascade on the same model (PR5)

Thesis ([SPEC-DARU.md](https://github.com/Grazinascito/adaptive-cascade-routing/blob/benchmark-daru/SPEC-DARU.md)): the cost savings of iFood's hybrid architecture (Sugar) also exist when we swap "small model vs large model" for "System 1 vs System 2 on the same model" (`Qwen/Qwen3-4B-AWQ`, single vLLM server):

- **SC-A — System 1 alone**: 1-token decision on `/v1/systemone` (fast, cheap).
- **SC-B — System 2 alone**: CoT + strict JSON on `/v1/chat/completions` (slow, expensive).
- **SC-C — cascade**: System 1 first; if P_top < α, the item escalates to System 2 (RouteLLM's R_α rule).

Dataset: DARU (18 largest Brazilian retailers) → 12 food/beverage categories, validation 200 + frozen test 600, tail = description with ≤ 20 characters.

Notebook pipeline: GPU check → dependencies → repo clone (`benchmark-daru`) → dataset via kagglehub → vLLM server → **pilot** (100-item gate; the notebook stops here if it fails) → full run (validation + α sweep + frozen test) → charts → artifact download.

> Verification: 86 tests + ruff + mypy green in the CI of [PR #7](https://github.com/Grazinascito/adaptive-cascade-routing/pull/7). This notebook runs real components only — a local vLLM server and the real DARU dataset; no step uses simulated data or responses.


In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0 or "T4" not in (gpu.stdout + gpu.stderr):
    raise RuntimeError(
        "No T4 GPU available. In 'Runtime > Change runtime type', select "
        "GPU T4 and run the notebook again."
    )
print("GPU check: T4 present")


In [ ]:
%pip install -q vllm httpx pyyaml pandas kagglehub openpyxl matplotlib

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Grazinascito/adaptive-cascade-routing.git"
BRANCH = "benchmark-daru"
REPO_DIR = Path("adaptive-cascade-routing").resolve()

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
rev = subprocess.run(
    ["git", "log", "--oneline", "-1"], cwd=REPO_DIR, capture_output=True, text=True
)
print(rev.stdout)

In [ ]:
import glob

import kagglehub

dataset_root = kagglehub.dataset_download(
    "gilsileydaru/classificao-produtos-varejo-cpg-prbr"
)
xlsx_matches = glob.glob(
    f"{dataset_root}/**/DATASETPRODUTOSVAREJOBRASIL.xlsx", recursive=True
)
if not xlsx_matches:
    raise FileNotFoundError(f"DARU xlsx not found under {dataset_root}")
print("dataset cached at:", xlsx_matches[0])


## vLLM server — Qwen3-4B-AWQ

`src.engine.server` starts the server as a subprocess with prefix caching and 90% of VRAM (≤ 14.4 GB of the T4's 16 GB). The health check polls `GET /v1/models` until it answers 200; on the first run vLLM downloads ~2.7 GB of weights — hence the 1800 s startup timeout. If anything fails, the full log is in `vllm_server.log`.


In [ ]:
import sys

import httpx

sys.path.insert(0, str(REPO_DIR))

from src.benchmark.runner import load_config
from src.engine import server

config = load_config(REPO_DIR / "config" / "benchmark_config.yaml")
port = config["serving"]["port"]
base_url = f"http://127.0.0.1:{port}"

print("server argv:", " ".join(server.build_server_command(config["model"]["id"], port)))
print("starting vLLM (first AWQ load on T4 takes a few minutes)...")
server.start_server(config["model"]["id"], port=port, startup_timeout_s=1800.0)
print("health:", server.health_check(base_url))
print("models:", httpx.get(f"{base_url}/v1/models", timeout=10).json())


## Pilot — the 100-item gate (SPEC-DARU §6)

100 validation items (70 head + 30 tail), System 1 and System 2 measured once. The gate: System 2's tail accuracy must beat System 1's by **≥ 10 percentage points** — this is the ASSIN 2 failure we want to catch in 1 hour, not after the full GPU run. If the gate fails, the cell **stops the notebook**.


In [ ]:
import subprocess
import sys

pilot = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.benchmark.run",
        "--config",
        "config/benchmark_config.yaml",
        "--pilot",
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(pilot.stdout)
if pilot.returncode != 0:
    print(pilot.stderr)
    raise RuntimeError(
        "PILOT FAILED: System 2 did not beat System 1 on the tail by >= 10 pp "
        "(SPEC-DARU §6). The notebook stops here — an honest diagnosis instead "
        "of an invalid full run."
    )
print("PILOT GATE PASSED — proceeding to full run.")


## Full run — validation + α sweep + frozen test

1. **Validation (200 items)**: System 1 and System 2 measured exactly once; the α sweep (0.70–0.90) is pure arithmetic over the saved outputs. Best α = highest cascade accuracy; ties broken by lowest token cost.
2. **Frozen test (600 items, measured 1x)**: SC-A, SC-B and SC-C with the chosen α, with 5 warmup requests per engine before each scenario.

Raw artifact: `results/raw_benchmark_data.json` (meta, per-item scenarios, sweep, summaries and the three gates: tail gap ≥ 10 pp, APGR ≥ 0.5, tokens ≤ 40% of System 2).


In [ ]:
import json
import subprocess
import sys

full = subprocess.run(
    [sys.executable, "-m", "src.benchmark.run", "--config", "config/benchmark_config.yaml"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
print(full.stdout)
if full.returncode != 0:
    print(full.stderr)
    raise RuntimeError("full run failed; see the output above.")

RESULTS = json.loads(
    (REPO_DIR / "results" / "raw_benchmark_data.json").read_text(encoding="utf-8")
)
print("result keys:", list(RESULTS))
for name, summary in RESULTS["summaries"].items():
    if name != "VALIDATION":
        keys = ("n_items", "accuracy", "accuracy_tail",
                "routed_to_s2_count", "decode_tokens_total")
        print(name, {k: summary[k] for k in keys})


## Charts

1. **Cost × accuracy frontier**: total decode tokens vs accuracy per scenario — the thesis holds if SC-C lands close to SC-B in accuracy at a fraction of the tokens.
2. **α sweep curve** on validation: cascade accuracy and routing rate vs α (the higher the α, the more items escalate to System 2).


In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

scenarios = {n: s for n, s in RESULTS["summaries"].items() if n != "VALIDATION"}

fig, ax = plt.subplots(figsize=(7, 5))
for name, summary in scenarios.items():
    ax.scatter(summary["decode_tokens_total"], summary["accuracy"], s=120)
    ax.annotate(
        name,
        (summary["decode_tokens_total"], summary["accuracy"]),
        xytext=(6, 4),
        textcoords="offset points",
    )
ax.set_xlabel("total decode tokens (cost)")
ax.set_ylabel("accuracy")
ax.set_title("Cost × accuracy frontier per scenario (frozen test)")
ax.grid(True, alpha=0.3)
fig.savefig(REPO_DIR / "results" / "cost_accuracy_frontier.png", dpi=150, bbox_inches="tight")
print("saved: results/cost_accuracy_frontier.png")

sweep = RESULTS["alpha_sweep"]["alphas"]
alphas = sorted(sweep, key=float)
accuracies = [sweep[a]["accuracy"] for a in alphas]
routed = [sweep[a]["routed_rate"] for a in alphas]

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(alphas, accuracies, "o-", label="cascade accuracy")
ax1.set_xlabel("α (P_top threshold)")
ax1.set_ylabel("accuracy")
ax2 = ax1.twinx()
ax2.plot(alphas, routed, "s--", color="tab:orange", label="routing rate")
ax2.set_ylabel("routing rate (fraction escalated to S2)")
ax1.set_title(f"α sweep on validation (best α = {RESULTS['alpha_sweep']['best_alpha']})")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="center left")
ax2.legend(loc="lower right")
fig.savefig(REPO_DIR / "results" / "alpha_sweep.png", dpi=150, bbox_inches="tight")
print("saved: results/alpha_sweep.png")
plt.show()


In [ ]:
import shutil

zip_path = shutil.make_archive(
    "results_daru_benchmark", "zip", root_dir=REPO_DIR / "results"
)
print("archive:", zip_path)
print("contents:", sorted(p.name for p in (REPO_DIR / "results").iterdir()))

try:
    from google.colab import files

    files.download(zip_path)
except ImportError:
    print("outside Colab: download the archive from the path above.")